In [2]:
%%writefile hotels.json
[
{
"hotel_id": 201,
"hotel_name": "Pearl Grand",
"city": "Hyderabad",
"category": "Business",
"rating": 4.4,
"rooms_available": 25,
"price_per_night": 4500,
"amenities": ["wifi", "breakfast", "gym"],
"contact": {
"phone": "9876500011",
"email": "pearlgrand@mail.com"
}
},
{
"hotel_id": 202,
"hotel_name": "Marina Bay Stay",
"city": "Dubai",
"category": "Luxury",
"rating": 4.8,
"rooms_available": 12,
"price_per_night": 18000,
"amenities": ["wifi", "pool", "spa", "sea_view"],
"contact": {
"phone": "9876500012",
"email": "marinabay@mail.com"
}
},
{
"hotel_id": 203,
"hotel_name": "Budget Inn",
"city": "Delhi",
"category": "Budget",
"rating": 3.9,
"rooms_available": 40,
"price_per_night": 2200,
"amenities": ["wifi"],
"contact": {
"phone": null,
"email": "budgetinn@mail.com"
}
},
{
"hotel_id": 204,
"hotel_name": "Hill View Resort",
"city": "Kochi",
"category": "Resort",
"rating": 4.5,
"rooms_available": 18,
"price_per_night": 7500,
"amenities": ["wifi", "breakfast", "pool"],
"contact": {
"phone": "9876500014",
"email": null
}
},
{
"hotel_id": 205,
"hotel_name": "Skyline Suites",
"city": "London",
"category": "Luxury",
"rating": 4.7,
"rooms_available": 8,
"price_per_night": 22000,
"amenities": ["wifi", "breakfast", "spa"],
"contact": {
"phone": "9876500015",
"email": "skyline@mail.com"
}
}
]

Writing hotels.json


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, array_contains

spark = SparkSession.builder.appName("HotelsJSON").getOrCreate()

hotels_df = spark.read.option(
    "multiline",
    "true"
).json("hotels.json")

hotels_df.show(truncate=False)
hotels_df.printSchema()

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [4]:
hotels_df.show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [5]:
hotels_df.select("hotel_name", "city", "rating").show()

+----------------+---------+------+
|      hotel_name|     city|rating|
+----------------+---------+------+
|     Pearl Grand|Hyderabad|   4.4|
| Marina Bay Stay|    Dubai|   4.8|
|      Budget Inn|    Delhi|   3.9|
|Hill View Resort|    Kochi|   4.5|
|  Skyline Suites|   London|   4.7|
+----------------+---------+------+



In [6]:
hotels_df.filter(col("category") == "Luxury").show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [7]:
hotels_df.filter(col("rating") > 4.5).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [8]:
hotels_df.filter(col("rooms_available") > 15).show(truncate=False)

+-----------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities              |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+-----------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym] |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi]                 |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]|Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500           |4.5   |18             |
+-----------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------

In [9]:
hotels_df.filter(col("price_per_night") > 10000).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [10]:
hotels_df.filter(col("city").isin("Dubai", "London")).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [11]:
hotels_df.filter(col("contact.phone").isNull()).show(truncate=False)

+---------+--------+-----+--------------------------+--------+----------+---------------+------+---------------+
|amenities|category|city |contact                   |hotel_id|hotel_name|price_per_night|rating|rooms_available|
+---------+--------+-----+--------------------------+--------+----------+---------------+------+---------------+
|[wifi]   |Budget  |Delhi|{budgetinn@mail.com, NULL}|203     |Budget Inn|2200           |3.9   |40             |
+---------+--------+-----+--------------------------+--------+----------+---------------+------+---------------+



In [12]:
hotels_df.filter(col("contact.email").isNull()).show(truncate=False)

+-----------------------+--------+-----+------------------+--------+----------------+---------------+------+---------------+
|amenities              |category|city |contact           |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+-----------------------+--------+-----+------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, pool]|Resort  |Kochi|{NULL, 9876500014}|204     |Hill View Resort|7500           |4.5   |18             |
+-----------------------+--------+-----+------------------+--------+----------------+---------------+------+---------------+



In [13]:
hotels_df.select(
    "hotel_name",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
).show(truncate=False)

+----------------+----------+-------------------+
|hotel_name      |phone     |email              |
+----------------+----------+-------------------+
|Pearl Grand     |9876500011|pearlgrand@mail.com|
|Marina Bay Stay |9876500012|marinabay@mail.com |
|Budget Inn      |NULL      |budgetinn@mail.com |
|Hill View Resort|9876500014|NULL               |
|Skyline Suites  |9876500015|skyline@mail.com   |
+----------------+----------+-------------------+



In [14]:
hotels_df.filter(array_contains(col("amenities"), "wifi")).show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [15]:
hotels_df.filter(array_contains(col("amenities"), "spa")).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [16]:
hotels_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Kochi|    1|
|   London|    1|
|    Delhi|    1|
|Hyderabad|    1|
|    Dubai|    1|
+---------+-----+



In [17]:
hotels_df.groupBy("category").count().show()

+--------+-----+
|category|count|
+--------+-----+
|  Resort|    1|
|  Budget|    1|
|Business|    1|
|  Luxury|    2|
+--------+-----+



In [18]:
hotels_df.groupBy("category").agg(avg("rating").alias("average_rating")).show()

+--------+--------------+
|category|average_rating|
+--------+--------------+
|  Resort|           4.5|
|  Budget|           3.9|
|Business|           4.4|
|  Luxury|          4.75|
+--------+--------------+



In [19]:
hotels_df.groupBy("city").agg(avg("price_per_night").alias("average_price")).show()

+---------+-------------+
|     city|average_price|
+---------+-------------+
|    Kochi|       7500.0|
|   London|      22000.0|
|    Delhi|       2200.0|
|Hyderabad|       4500.0|
|    Dubai|      18000.0|
+---------+-------------+



In [20]:
hotels_df.agg(max("price_per_night").alias("highest_price_per_night")).show()

+-----------------------+
|highest_price_per_night|
+-----------------------+
|                  22000|
+-----------------------+



In [21]:
hotels_df = hotels_df.withColumn(
    "total_potential_revenue",
    col("rooms_available") * col("price_per_night")
)

hotels_df.show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|total_potential_revenue|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |112500                 |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |216000                 |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40       

In [22]:
hotels_df.orderBy(col("rating").desc()).show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|total_potential_revenue|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |216000                 |
|[wifi, breakfast, spa]     |Luxury  |London   |{skyline@mail.com, 9876500015}   |205     |Skyline Suites  |22000          |4.7   |8              |176000                 |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500           |4.5   |18       

In [23]:
hotels_flattened_df = hotels_df.select(
    "hotel_id",
    "hotel_name",
    "city",
    "category",
    "rating",
    "rooms_available",
    "price_per_night",
    "total_potential_revenue",
    "amenities",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

hotels_flattened_df.write.mode("overwrite").parquet("hotels_flattened.parquet")